# 面试问题：怎样设计可靠的 Agent Tool Calling，而不是直接执行模型输出？

**一句话回答**：把模型输出视为不可信调用提议。宿主程序维护最小工具注册表，严格解析 schema，绑定用户身份与租户做授权；读操作设置超时/重试，写操作使用 idempotency key、审批和结果账本；任何工具返回也作为不可信 observation，最后用状态机记录 propose→validate→authorize→approve→execute→observe。

本 Notebook 用标准库实现 JSON-schema 子集、权限、幂等账本、重试与审计 trace，不依赖 Agent 框架。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib, json, time  # 导入本单元所需的依赖。

SEED106=10601  # 计算并保存当前步骤的中间状态。
assert SEED106==10601  # 用受控断言验证关键不变量。
assert json.loads('{"x":1}')["x"]==1  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"call-1").hexdigest()!=hashlib.sha256(b"call-2").hexdigest()  # 用受控断言验证关键不变量。

## 1. 工具注册表是能力边界

每个工具声明名称、参数 schema、是否写操作、所需 scope、超时和最大重试。不要暴露通用 shell/SQL/HTTP 作为“万能工具”；按业务拆成窄接口，宿主可独立鉴权和审计。模型看到的描述不是授权凭证。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class ToolSpec106:  # 定义承载本节状态与行为的数据结构。
    name:str; schema:dict; write:bool; scope:str; timeout_ms:int=1000; retries:int=0  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.name or self.timeout_ms<=0 or self.retries<0: raise ValueError("tool_contract")  # 按当前条件选择后续控制路径。
registry106={  # 计算并保存当前步骤的中间状态。
    "get_balance":ToolSpec106("get_balance",{"required":["account_id"],"properties":{"account_id":"string"}},False,"balance:read",500,1),  # 执行当前语句以推进本节示例。
    "transfer":ToolSpec106("transfer",{"required":["to","amount"],"properties":{"to":"string","amount":"number","currency":"string"}},True,"transfer:write",800,0)}  # 执行当前语句以推进本节示例。
assert set(registry106)=={"get_balance","transfer"}  # 用受控断言验证关键不变量。
assert registry106["transfer"].write and registry106["get_balance"].retries==1  # 用受控断言验证关键不变量。
try: ToolSpec106("",{},False,"x",0); raise AssertionError("bad tool accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="tool_contract"  # 捕获预期异常并验证失败分支。

## 2. 严格 schema：拒绝未知字段与隐式类型转换

模型可能输出缺字段、字符串金额、额外命令或畸形 JSON。解析层限定长度、required、type 和 additional properties；不要把 `"100"` 悄悄转换为数字，因为模糊修复会扩大攻击面。失败后可把结构化错误反馈给模型有限重试。

In [ ]:
TYPE106={"string":str,"number":(int,float)}  # 计算并保存当前步骤的中间状态。
def validate_args106(args,schema):  # 定义本节可复用的核心函数。
    if not isinstance(args,dict): return False,"not_object"  # 按当前条件选择后续控制路径。
    required=set(schema.get("required",[])); props=schema.get("properties",{})  # 计算并保存当前步骤的中间状态。
    if not required<=set(args): return False,"missing_required"  # 按当前条件选择后续控制路径。
    if set(args)-set(props): return False,"unknown_property"  # 按当前条件选择后续控制路径。
    for k,v in args.items():  # 遍历输入元素以累积或检查结果。
        if isinstance(v,bool) and props[k]=="number": return False,"wrong_type"  # 按当前条件选择后续控制路径。
        if not isinstance(v,TYPE106[props[k]]): return False,"wrong_type"  # 按当前条件选择后续控制路径。
    return True,"valid"  # 返回当前分支计算出的结果。
assert validate_args106({"to":"u2","amount":12.5},registry106["transfer"].schema)==(True,"valid")  # 用受控断言验证关键不变量。
assert validate_args106({"to":"u2","amount":"12.5"},registry106["transfer"].schema)==(False,"wrong_type")  # 用受控断言验证关键不变量。
assert validate_args106({"to":"u2","amount":1,"shell":"rm"},registry106["transfer"].schema)==(False,"unknown_property")  # 用受控断言验证关键不变量。

## 3. 模型输出是 Proposal，不是命令

Proposal 至少包含 call ID、工具名和参数。宿主先 JSON 解析与大小限制，再查注册表和 schema；未知工具直接拒绝。解析错误不能执行“最接近”的工具。call ID 由宿主生成或重新绑定，不能完全信任模型提供的键。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Proposal106: call_id:str; tool:str; args:dict  # 定义承载本节状态与行为的数据结构。
def parse_proposal106(raw,max_chars=1000):  # 定义本节可复用的核心函数。
    if not isinstance(raw,str) or len(raw)>max_chars: raise ValueError("proposal_size")  # 按当前条件选择后续控制路径。
    try: obj=json.loads(raw)  # 尝试执行可能失败的受控操作。
    except json.JSONDecodeError: raise ValueError("proposal_json")  # 捕获预期异常并验证失败分支。
    if set(obj)!={"call_id","tool","args"} or obj["tool"] not in registry106: raise ValueError("proposal_shape")  # 按当前条件选择后续控制路径。
    ok,reason=validate_args106(obj["args"],registry106[obj["tool"]].schema)  # 计算并保存当前步骤的中间状态。
    if not ok: raise ValueError(reason)  # 按当前条件选择后续控制路径。
    return Proposal106(obj["call_id"],obj["tool"],obj["args"])  # 返回当前分支计算出的结果。
proposal106=parse_proposal106('{"call_id":"c1","tool":"get_balance","args":{"account_id":"a1"}}')  # 计算并保存当前步骤的中间状态。
assert proposal106.tool=="get_balance"  # 用受控断言验证关键不变量。
try: parse_proposal106('{"call_id":"c2","tool":"shell","args":{}}'); raise AssertionError("unknown tool accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="proposal_shape"  # 捕获预期异常并验证失败分支。
assert proposal106.call_id=="c1" and proposal106.args=={"account_id":"a1"}  # 用受控断言验证关键不变量。

## 4. 授权由下游基于真实身份完成

Prompt 中写“不能跨租户”不是安全控制。执行器用认证会话绑定 user/tenant/scopes，并验证资源归属；模型不能自行声明身份。高风险写操作再要求短期 approval token，且批准页面展示规范化参数而非模型生成说明。

In [ ]:
accounts106={"a1":{"tenant":"T1","balance":100.},"a2":{"tenant":"T2","balance":900.}}  # 计算并保存当前步骤的中间状态。
def authorize106(proposal,identity):  # 定义本节可复用的核心函数。
    spec=registry106[proposal.tool]  # 计算并保存当前步骤的中间状态。
    if spec.scope not in identity["scopes"]: return False,"missing_scope"  # 按当前条件选择后续控制路径。
    account=proposal.args.get("account_id")  # 计算并保存当前步骤的中间状态。
    if account and (account not in accounts106 or accounts106[account]["tenant"]!=identity["tenant"]): return False,"tenant_boundary"  # 按当前条件选择后续控制路径。
    return True,"allowed"  # 返回当前分支计算出的结果。
ident106={"user":"u1","tenant":"T1","scopes":{"balance:read"}}  # 计算并保存当前步骤的中间状态。
assert authorize106(proposal106,ident106)==(True,"allowed")  # 用受控断言验证关键不变量。
cross106=Proposal106("c2","get_balance",{"account_id":"a2"}); assert authorize106(cross106,ident106)==(False,"tenant_boundary")  # 计算并保存当前步骤的中间状态。
assert authorize106(Proposal106("c3","transfer",{"to":"u2","amount":1}),ident106)==(False,"missing_scope")  # 用受控断言验证关键不变量。

## 5. 写操作用幂等键和结果账本

网络超时不代表写入失败，盲目重试可能重复扣款。幂等键绑定租户、用户、工具和规范化参数；相同键相同请求返回已记录结果，相同键不同参数必须冲突。账本与业务写入应在同一事务或通过 outbox 协调。

In [ ]:
ledger106={}; transfers106=[]  # 计算并保存当前步骤的中间状态。
def canonical106(tool,args): return hashlib.sha256(json.dumps({"tool":tool,"args":args},sort_keys=True,separators=(",",":")).encode()).hexdigest()  # 定义本节可复用的核心函数。
def execute_transfer106(key,args):  # 定义本节可复用的核心函数。
    fp=canonical106("transfer",args)  # 计算并保存当前步骤的中间状态。
    if key in ledger106:  # 按当前条件选择后续控制路径。
        if ledger106[key]["fingerprint"]!=fp: raise ValueError("idempotency_conflict")  # 按当前条件选择后续控制路径。
        return ledger106[key]["result"],True  # 返回当前分支计算出的结果。
    result={"transfer_id":f"tx-{len(transfers106)+1}","status":"accepted"}; transfers106.append(args.copy()); ledger106[key]={"fingerprint":fp,"result":result}; return result,False  # 计算并保存当前步骤的中间状态。
tx1_106,replay1_106=execute_transfer106("idem-1",{"to":"u2","amount":5}); tx2_106,replay2_106=execute_transfer106("idem-1",{"to":"u2","amount":5})  # 计算并保存当前步骤的中间状态。
assert tx1_106==tx2_106 and not replay1_106 and replay2_106  # 用受控断言验证关键不变量。
assert len(transfers106)==1  # 用受控断言验证关键不变量。
try: execute_transfer106("idem-1",{"to":"u2","amount":6}); raise AssertionError("conflict accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="idempotency_conflict"  # 捕获预期异常并验证失败分支。

## 6. 重试策略取决于读写语义

可重试错误限于明确的 timeout/temporary failure，并使用指数退避与总 deadline。读操作通常可重试；非幂等写若没有业务幂等合同不可自动重试。下面的假工具在第二次成功，用确定性时钟记录尝试次数。

In [ ]:
class Temporary106(Exception): pass  # 定义承载本节状态与行为的数据结构。
attempts106={"n":0}  # 计算并保存当前步骤的中间状态。
def flaky_read106():  # 定义本节可复用的核心函数。
    attempts106["n"]+=1  # 计算并保存当前步骤的中间状态。
    if attempts106["n"]<2: raise Temporary106("timeout")  # 按当前条件选择后续控制路径。
    return {"balance":100.}  # 返回当前分支计算出的结果。
def retry106(fn,max_retries):  # 定义本节可复用的核心函数。
    errors=[]  # 计算并保存当前步骤的中间状态。
    for attempt in range(max_retries+1):  # 遍历输入元素以累积或检查结果。
        try: return fn(),attempt,errors  # 尝试执行可能失败的受控操作。
        except Temporary106 as e:  # 捕获预期异常并验证失败分支。
            errors.append(str(e))  # 执行当前语句以推进本节示例。
            if attempt==max_retries: raise  # 按当前条件选择后续控制路径。
result106,used_retries106,errors106=retry106(flaky_read106,1)  # 计算并保存当前步骤的中间状态。
assert result106=={"balance":100.} and used_retries106==1  # 用受控断言验证关键不变量。
assert attempts106["n"]==2  # 用受控断言验证关键不变量。
assert errors106==["timeout"]  # 用受控断言验证关键不变量。

## 7. 审批是绑定具体动作的 capability

“你确定吗？”不足够：审批 token 要绑定调用摘要、用户、过期时间和单次消费。展示目的地、金额与权限影响，不能只显示可能被注入的自然语言。审批后参数变化必须重新批准。

In [ ]:
approvals106={}  # 计算并保存当前步骤的中间状态。
def issue_approval106(user,tool,args,expires):  # 定义本节可复用的核心函数。
    digest=canonical106(tool,args); token=hashlib.sha256(f"{user}:{digest}:{expires}".encode()).hexdigest(); approvals106[token]={"user":user,"digest":digest,"expires":expires,"used":False}; return token  # 计算并保存当前步骤的中间状态。
def consume_approval106(token,user,tool,args,now):  # 定义本节可复用的核心函数。
    row=approvals106.get(token); digest=canonical106(tool,args)  # 计算并保存当前步骤的中间状态。
    if not row or row["used"] or row["user"]!=user or row["digest"]!=digest or now>row["expires"]: return False  # 按当前条件选择后续控制路径。
    row["used"]=True; return True  # 计算并保存当前步骤的中间状态。
args106={"to":"u2","amount":5}; token106=issue_approval106("u1","transfer",args106,100)  # 计算并保存当前步骤的中间状态。
assert consume_approval106(token106,"u1","transfer",args106,99)  # 用受控断言验证关键不变量。
assert not consume_approval106(token106,"u1","transfer",args106,99)  # 用受控断言验证关键不变量。
assert not consume_approval106(issue_approval106("u1","transfer",args106,100),"u1","transfer",{"to":"u2","amount":6},99)  # 用受控断言验证关键不变量。

## 8. 用显式状态机和 trace 验收

每次调用记录 proposal、校验、授权、审批、执行、结果摘要、错误类型与耗时，敏感字段脱敏。离线回放要覆盖畸形 JSON、越权、重复写、超时、工具返回注入和取消；指标包括任务成功率、非法调用率、人工审批率和每任务工具步数。

In [ ]:
trace106=[]  # 计算并保存当前步骤的中间状态。
def record106(state,call_id,detail): trace106.append({"seq":len(trace106),"state":state,"call_id":call_id,"detail":detail})  # 定义本节可复用的核心函数。
for state,detail in [("proposed","get_balance"),("validated","schema_ok"),("authorized","scope_ok"),("executed","success"),("observed","redacted_result")]: record106(state,"c1",detail)  # 遍历输入元素以累积或检查结果。
manifest106={"schema":1,"registry":[k for k in sorted(registry106)],"unknown_fields":"reject","writes":"approval+idempotency","trace_states":[r["state"] for r in trace106]}; digest106=hashlib.sha256(json.dumps(manifest106,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert [r["seq"] for r in trace106]==list(range(5))  # 用受控断言验证关键不变量。
assert manifest106["trace_states"]==["proposed","validated","authorized","executed","observed"]  # 用受控断言验证关键不变量。
assert len(digest106)==64 and manifest106["unknown_fields"]=="reject"  # 用受控断言验证关键不变量。

## 面试总结

完整链路是 **最小注册表 → 严格 schema → Proposal → 身份/租户授权 → 写操作审批 → 幂等账本 → 有语义的重试 → observation 隔离与 trace**。安全边界必须在确定性宿主和下游系统，而不能依赖模型“记得守规则”。

延伸阅读：[OWASP Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)、[JSON Schema 规范](https://json-schema.org/specification)、[Toolformer](https://arxiv.org/abs/2302.04761)。